<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/01_pytorch_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from typing import Tuple
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# ==========================================
# --- 1. Custom Data Pipeline (Tasks 1 & 2) ---
# ==========================================
# This class demonstrates dynamic disk-loading to prevent RAM overflow on large datasets.

target_dir = 'data/data/fashion/unzipped'

class FashionMNIST(Dataset):
    """Custom Dataset that dynamically loads image arrays from disk."""
    def __init__(self, indices, labels):
        self.indices = indices
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        file_idx = int(self.indices[idx])
        img_path = os.path.join(target_dir, f'img_{file_idx}.npy')

        # Load and normalize
        X_raw = np.load(img_path)
        X = (X_raw.astype(np.float32) / 255.0).flatten()
        y = int(self.labels[idx])

        return X, y

    @staticmethod
    def create_split(fraction_train, fraction_validation, fraction_test, stratify=True, shuffle=True):
        """Creates Stratified Train, Validation, and Test splits."""
        full_labels = np.load(os.path.join(target_dir, 'labels.npy')).flatten()
        all_indices = np.arange(len(full_labels))
        use_shuffle = True if stratify else shuffle

        # Test vs Train+Val
        if fraction_test > 0.0:
            train_val_idx, test_idx = train_test_split(
                all_indices, test_size=fraction_test,
                stratify=full_labels if stratify else None,
                shuffle=use_shuffle, random_state=42
            )
        else:
            train_val_idx, test_idx = all_indices, np.array([], dtype=int)

        # Train vs Validation
        if fraction_validation > 0.0:
            val_ratio = fraction_validation / (fraction_train + fraction_validation)
            train_idx, val_idx = train_test_split(
                train_val_idx, test_size=val_ratio,
                stratify=full_labels[train_val_idx] if stratify else None,
                shuffle=use_shuffle, random_state=42
            )
        else:
            train_idx, val_idx = train_val_idx, np.array([], dtype=int)

        return (
            FashionMNIST(train_idx, full_labels[train_idx]),
            FashionMNIST(val_idx, full_labels[val_idx]),
            FashionMNIST(test_idx, full_labels[test_idx])
        )

# ==========================================
# --- 2. Architecture Definition (Task 3) ---
# ==========================================

class OneLayerNetwork(nn.Module):
    """A simple linear neural network for 10-class classification."""
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(784, 10) # 28x28 pixels -> 10 classes

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.layer(x)  # Outputs raw logits for CrossEntropyLoss

# ==========================================
# --- 3. Training Infrastructure (Task 4) ---
# ==========================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def epoch(model, loader_train, optimizer, loss_fn):
    """Executes a single training epoch."""
    model.train()
    losses = []
    for X_batch, y_batch in loader_train:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)

def evaluate(model, loader_valid, loss_fn=None):
    """Evaluates the model and computes macro F1-score."""
    model.eval()
    predictions, labels, losses = [], [], []

    with torch.no_grad():
        for X_batch, y_batch in loader_valid:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)

            labels.extend(y_batch.cpu().numpy())
            predictions.extend(y_pred.argmax(dim=1).cpu().numpy())

            if loss_fn is not None:
                losses.append(loss_fn(y_pred, y_batch).item())

    f1 = f1_score(labels, predictions, average='macro')
    if loss_fn is None:
        return {'f1': f1}
    return {'loss': np.mean(losses), 'f1': f1}

def fit(model, loader_train, loader_valid, epochs, lr):
    """Orchestrates the training and validation loops."""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for i in range(epochs):
        loss_train = epoch(model, loader_train, optimizer, loss_fn)
        metrics = evaluate(model, loader_valid, loss_fn)
        history.append({
            'loss_train': loss_train,
            'loss_valid': metrics['loss'],
            'f1_valid': metrics['f1']
        })
        if (i + 1) % 10 == 0 or i == 0:
            print(f"Epoch {i+1:02d}/{epochs} | Train Loss: {loss_train:.4f} | Valid Loss: {metrics['loss']:.4f} | Valid F1: {metrics['f1']:.4f}")

    return pd.DataFrame(history)

# Note: Execution calls requiring downloaded data are omitted for the portfolio presentation
# to ensure it can be read cleanly as an architectural showcase.